# Batch Attribution: Production-Scale Interpretability

**Learning objectives:**
- Compute attributions efficiently over large datasets using native batch processing
- Implement attribution caching to avoid redundant computation
- Build a minimal FastAPI interpretability service and test it programmatically
- Compare latency across attribution methods at batch scale
- Generate an attribution audit report for a set of predictions

**Estimated time:** 15 minutes

**Model:** ResNet-18 pretrained (ImageNet-1K)

In [ ]:
learning_objectives(["- Compute attributions efficiently over large datasets using native batch processing", "Implement attribution caching to avoid redundant computation", "Build a minimal FastAPI interpretability service and test it programmatically", "Compare latency across attribution methods at batch scale", "Generate an attribution audit report for a set of predictions", "15 minutes"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()

In [ ]:
import torch
import numpy as np
import time
import json
import hashlib
import warnings
warnings.filterwarnings('ignore')

import torchvision.transforms as T
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from captum.attr import (
    IntegratedGradients, Saliency, GradientShap,
    LayerGradCam, LayerAttribution
)

print('Imports ready.')

## 1. Load Model and Data

We use CIFAR-10 resized to 224×224 with a pretrained ResNet-18. The model was trained on ImageNet, so predictions will not be semantically correct — but attributions will still show meaningful spatial patterns.

In [ ]:
# Load pretrained ResNet-18
weights = ResNet18_Weights.IMAGENET1K_V1
model = resnet18(weights=weights)
model.eval()

# Preprocessing for CIFAR-10 images fed to ImageNet model
transform = T.Compose([
    T.Resize(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

# CIFAR-10 class names
CIFAR_CLASSES = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

# Load a small subset of CIFAR-10
dataset = CIFAR10(root='/tmp/cifar10', train=False, download=True, transform=transform)
subset = Subset(dataset, indices=list(range(200)))  # 200 examples
dataloader = DataLoader(subset, batch_size=1, shuffle=False)

print(f'Model: ResNet-18 (pretrained ImageNet-1K)')
print(f'Dataset: CIFAR-10 test set, {len(subset)} examples')
print(f'Image shape after transform: 3×224×224')

## 2. Batch Attribution with Single Method

Process an entire dataset and collect per-example attributions and predictions.

In [ ]:
def batch_saliency_attribution(
    model,
    dataloader,
    n_examples: int = 100,
    abs_val: bool = True,
):
    """
    Compute Saliency attributions for a batch of examples.
    Returns list of dicts with attrs, pred_class, confidence, label.
    """
    saliency = Saliency(lambda x: model(x))
    results = []

    for i, (images, labels) in enumerate(dataloader):
        if i >= n_examples:
            break

        with torch.no_grad():
            logits = model(images)
            probs = torch.softmax(logits, dim=1)[0]
            pred_class = probs.argmax().item()
            confidence = probs[pred_class].item()

        # Saliency requires gradient — enable grad for input
        images_req = images.requires_grad_(True)
        attrs = saliency.attribute(images_req, target=pred_class, abs=abs_val)

        results.append({
            'attrs': attrs.detach().sum(dim=1).squeeze(0),  # (H, W)
            'pred_class': pred_class,
            'confidence': confidence,
            'true_label': labels.item(),
            'correct': pred_class == labels.item(),
        })

    return results


print('Running Saliency attribution on 100 CIFAR-10 examples...')
t0 = time.perf_counter()
saliency_results = batch_saliency_attribution(model, dataloader, n_examples=100)
t1 = time.perf_counter()

elapsed = t1 - t0
n_correct = sum(r['correct'] for r in saliency_results)

print(f'Done. {len(saliency_results)} examples in {elapsed:.1f}s')
print(f'Throughput: {len(saliency_results)/elapsed:.1f} examples/sec')
print(f'Time per example: {elapsed/len(saliency_results)*1000:.1f} ms')
print(f'Model accuracy (ImageNet→CIFAR): {n_correct}/{len(saliency_results)} = {n_correct/len(saliency_results):.1%}')
print('(Low accuracy expected: model was not fine-tuned on CIFAR)')

## 3. Attribution Caching

Caching avoids recomputing attributions for identical inputs. The cache key is an MD5 hash of the input tensor bytes plus method and target parameters.

In [ ]:
class AttributionCache:
    """Simple in-memory LRU attribution cache."""

    def __init__(self, max_size: int = 512):
        self._store = {}       # key → result
        self._order = []       # insertion order for LRU eviction
        self._max_size = max_size
        self._hits = 0
        self._misses = 0

    def _key(self, inputs: torch.Tensor, method: str, target: int) -> str:
        raw = inputs.cpu().numpy().tobytes()
        return hashlib.md5(raw + method.encode() + str(target).encode()).hexdigest()

    def get(self, inputs: torch.Tensor, method: str, target: int):
        key = self._key(inputs, method, target)
        if key in self._store:
            self._hits += 1
            return self._store[key]
        self._misses += 1
        return None

    def set(self, inputs: torch.Tensor, method: str, target: int, value):
        key = self._key(inputs, method, target)
        if len(self._order) >= self._max_size:
            oldest = self._order.pop(0)
            self._store.pop(oldest, None)
        self._store[key] = value
        self._order.append(key)

    @property
    def hit_rate(self):
        total = self._hits + self._misses
        return self._hits / total if total > 0 else 0.0

    def stats(self) -> dict:
        return {'hits': self._hits, 'misses': self._misses,
                'hit_rate': f'{self.hit_rate:.1%}',
                'size': len(self._store)}


cache = AttributionCache(max_size=256)

def cached_saliency(model, inputs, target, cache: AttributionCache):
    """Saliency attribution with caching."""
    cached = cache.get(inputs, 'saliency', target)
    if cached is not None:
        return cached

    saliency = Saliency(lambda x: model(x))
    attrs = saliency.attribute(inputs.requires_grad_(True), target=target, abs=True)
    result = attrs.detach().sum(dim=1).squeeze(0)
    cache.set(inputs, 'saliency', target, result)
    return result


# Run the same 50 examples twice to demonstrate caching
test_inputs = [b[0] for b in list(dataloader)[:50]]

# First pass: all cache misses
print('First pass (cold cache)...')
t0 = time.perf_counter()
for img in test_inputs:
    with torch.no_grad():
        pred = model(img).argmax(dim=1).item()
    cached_saliency(model, img, pred, cache)
t1 = time.perf_counter()
cold_time = t1 - t0
print(f'  Cold pass: {cold_time:.3f}s | {cache.stats()}')

# Second pass: all cache hits
print('Second pass (warm cache)...')
t0 = time.perf_counter()
for img in test_inputs:
    with torch.no_grad():
        pred = model(img).argmax(dim=1).item()
    cached_saliency(model, img, pred, cache)
t1 = time.perf_counter()
warm_time = t1 - t0
print(f'  Warm pass: {warm_time:.3f}s | {cache.stats()}')
print(f'  Speedup: {cold_time/warm_time:.0f}x')

## 4. Method Latency Benchmark

Compare wall-clock time per example across Saliency, GradCAM, and IG with varying step counts.

In [ ]:
# Prepare benchmark inputs
benchmark_images = []
benchmark_targets = []
for i, (img, lbl) in enumerate(dataloader):
    if i >= 20:
        break
    with torch.no_grad():
        pred = model(img).argmax(dim=1).item()
    benchmark_images.append(img)
    benchmark_targets.append(pred)

print(f'Benchmark set: {len(benchmark_images)} examples')

# Attribution methods to benchmark
methods_to_test = [
    ('Saliency', lambda img, tgt: Saliency(lambda x: model(x)).attribute(
        img.requires_grad_(True), target=tgt, abs=True)),
    ('GradCAM', lambda img, tgt: LayerGradCam(lambda x: model(x), model.layer4[-1]).attribute(
        img, target=tgt)),
    ('IG (n=20)', lambda img, tgt: IntegratedGradients(lambda x: model(x)).attribute(
        img, torch.zeros_like(img), target=tgt, n_steps=20)),
    ('IG (n=50)', lambda img, tgt: IntegratedGradients(lambda x: model(x)).attribute(
        img, torch.zeros_like(img), target=tgt, n_steps=50)),
    ('GradShap (n=10)', lambda img, tgt: GradientShap(lambda x: model(x)).attribute(
        img, torch.zeros_like(img).expand(10, -1, -1, -1), n_samples=10, target=tgt)),
]

timing_results = {}

for method_name, method_fn in methods_to_test:
    times = []
    for img, tgt in zip(benchmark_images, benchmark_targets):
        t0 = time.perf_counter()
        _ = method_fn(img, tgt)
        times.append(time.perf_counter() - t0)
    timing_results[method_name] = times
    print(f'  {method_name:<20s}: {np.mean(times)*1000:6.1f} ms/example  '
          f'(± {np.std(times)*1000:.1f} ms)')

In [ ]:
# Plot latency comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

method_names = list(timing_results.keys())
means = [np.mean(t) * 1000 for t in timing_results.values()]
stds  = [np.std(t) * 1000 for t in timing_results.values()]

# Bar chart: mean latency
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#9C27B0']
bars = axes[0].bar(method_names, means, yerr=stds, capsize=5,
                    color=colors[:len(method_names)], alpha=0.85, edgecolor='white')
axes[0].set_ylabel('Time per example (ms)', fontsize=11)
axes[0].set_title('Attribution Method Latency\n(ResNet-18, 224×224, CPU)', fontweight='bold')
axes[0].tick_params(axis='x', rotation=25)
axes[0].grid(axis='y', alpha=0.3)
for bar, mean in zip(bars, means):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                  f'{mean:.0f}ms', ha='center', fontsize=8)

# Box plot: distribution of latencies
axes[1].boxplot(
    [np.array(t) * 1000 for t in timing_results.values()],
    labels=method_names,
    patch_artist=True,
    boxprops=dict(facecolor='lightblue', alpha=0.7),
)
axes[1].set_ylabel('Time per example (ms)', fontsize=11)
axes[1].set_title('Latency Distribution\n(n=20 examples per method)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=25)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('attribution_latency_benchmark.png', dpi=130, bbox_inches='tight')
plt.show()
print('Figure saved: attribution_latency_benchmark.png')

## 5. Generating an Attribution Audit Report

For compliance and monitoring, generate a structured JSON report for each prediction.

In [ ]:
import uuid
from datetime import datetime

def generate_attribution_report(
    model,
    inputs: torch.Tensor,
    true_label: int,
    model_id: str = 'resnet18_imagenet',
    n_steps: int = 50,
) -> dict:
    """
    Generate a structured attribution audit report for one example.

    Returns a dict that can be serialized to JSON for compliance logging.
    """
    # Prediction
    with torch.no_grad():
        logits = model(inputs)
        probs = torch.softmax(logits, dim=1)[0]
        pred_class = probs.argmax().item()
        confidence = probs[pred_class].item()

    # IG attribution
    ig = IntegratedGradients(lambda x: model(x))
    baseline = torch.zeros_like(inputs)
    attrs, delta = ig.attribute(
        inputs, baseline,
        target=pred_class,
        n_steps=n_steps,
        return_convergence_delta=True,
    )

    # Spatial attribution statistics
    attr_map = attrs.abs().sum(dim=1).squeeze(0).detach().numpy()  # (H, W)
    H, W = attr_map.shape

    # Border vs. center attribution split
    border = int(H * 0.1)
    center_attr = attr_map[border:-border, border:-border].sum()
    border_attr = attr_map.sum() - center_attr
    border_fraction = float(border_attr / (attr_map.sum() + 1e-8))

    # Attribution sum for completeness check
    f_x = torch.softmax(logits, dim=1)[0, pred_class].item()
    f_b = torch.softmax(model(baseline), dim=1)[0, pred_class].item()

    # Input fingerprint
    input_hash = hashlib.sha256(inputs.cpu().numpy().tobytes()).hexdigest()[:16]

    return {
        'report_id': str(uuid.uuid4()),
        'timestamp': datetime.utcnow().isoformat() + 'Z',
        'model_id': model_id,
        'input_hash': input_hash,
        'true_label': true_label,
        'true_class': CIFAR_CLASSES[true_label],
        'predicted_class_index': pred_class,
        'predicted_confidence': round(confidence, 4),
        'correct': pred_class == true_label,
        'attribution': {
            'method': 'integrated_gradients',
            'n_steps': n_steps,
            'baseline': 'zero_image',
            'convergence_delta': round(float(delta.item()), 6),
            'completeness_satisfied': abs(delta.item()) < 0.05,
            'f_input': round(f_x, 4),
            'f_baseline': round(f_b, 4),
            'attribution_sum': round(float(attrs.sum().item()), 4),
        },
        'spatial_stats': {
            'border_attribution_fraction': round(border_fraction, 4),
            'max_attr_pixel': [int(x) for x in np.unravel_index(
                attr_map.argmax(), attr_map.shape)],
            'top_quartile_fraction': float(
                (attr_map > np.percentile(attr_map, 75)).mean()),
        },
    }


# Generate reports for first 5 examples
print('Generating attribution reports...')
reports = []
for i, (img, lbl) in enumerate(dataloader):
    if i >= 5:
        break
    report = generate_attribution_report(model, img, lbl.item(), n_steps=30)
    reports.append(report)
    print(f"  [{i+1}/5] {report['true_class']:<12s} → "
          f"pred_class={report['predicted_class_index']:3d} | "
          f"delta={report['attribution']['convergence_delta']:+.5f} | "
          f"complete={report['attribution']['completeness_satisfied']}")

print(f'\n{len(reports)} reports generated.')

In [ ]:
# Save reports to JSON
report_path = 'attribution_audit_reports.json'
with open(report_path, 'w') as f:
    json.dump(reports, f, indent=2)

print(f'Reports saved to: {report_path}')
print()
print('Sample report (first example):')
print(json.dumps(reports[0], indent=2))

## 6. Attribution Gallery: Correct vs. Incorrect Predictions

Compare attribution patterns for correctly and incorrectly classified examples.

In [ ]:
import matplotlib.cm as cm

# Separate correct and incorrect predictions from our batch results
correct_results = [r for r in saliency_results if r['correct']][:3]
incorrect_results = [r for r in saliency_results if not r['correct']][:3]

print(f'Found {len([r for r in saliency_results if r["correct"]])} correct predictions')
print(f'Found {len([r for r in saliency_results if not r["correct"]])} incorrect predictions')

# Re-fetch the input images for visualization
viz_data = []
for i, (img, lbl) in enumerate(dataloader):
    if i >= 100:
        break
    with torch.no_grad():
        pred = model(img).argmax(dim=1).item()
    attrs = Saliency(lambda x: model(x)).attribute(
        img.requires_grad_(True), target=pred, abs=True
    ).detach().sum(dim=1).squeeze(0)
    viz_data.append({'img': img, 'attrs': attrs, 'pred': pred,
                      'label': lbl.item(), 'correct': pred == lbl.item()})

correct_examples = [d for d in viz_data if d['correct']][:4]
incorrect_examples = [d for d in viz_data if not d['correct']][:4]

mean_t = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std_t  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def show_attribution_row(examples, row_label, axes_row):
    for j, ex in enumerate(examples):
        if j >= len(axes_row):
            break
        img_d = (ex['img'].squeeze(0) * std_t + mean_t).clamp(0, 1)
        img_np = img_d.permute(1, 2, 0).numpy()

        attr_np = ex['attrs'].numpy()
        attr_np = (attr_np - attr_np.min()) / (attr_np.max() - attr_np.min() + 1e-8)

        overlay = (0.6 * img_np + 0.4 * cm.hot(attr_np)[:, :, :3]).clip(0, 1)
        axes_row[j].imshow(overlay)
        axes_row[j].set_title(
            f'True: {CIFAR_CLASSES[ex["label"]]}\nPred: {ex["pred"]}',
            fontsize=7
        )
        axes_row[j].axis('off')


fig, axes = plt.subplots(2, 4, figsize=(14, 7))

show_attribution_row(correct_examples, 'Correct', axes[0])
show_attribution_row(incorrect_examples, 'Incorrect', axes[1])

axes[0][0].set_ylabel('Correct\npredictions', fontsize=10, rotation=0,
                        labelpad=60, va='center')
axes[1][0].set_ylabel('Incorrect\npredictions', fontsize=10, rotation=0,
                        labelpad=60, va='center')

fig.suptitle('Saliency Attribution Gallery: Correct vs. Incorrect Predictions\n'
              '(ResNet-18 on CIFAR-10, Saliency, overlay = image + heatmap)',
              fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('attribution_gallery.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved: attribution_gallery.png')

## 7. Throughput Scaling Analysis

Measure how throughput scales with batch size for Saliency (the fastest method).

In [ ]:
# Throughput vs. batch size for Saliency
batch_sizes = [1, 2, 4, 8]
throughputs = []

print('Throughput scaling analysis (Saliency):')
for bs in batch_sizes:
    dl = DataLoader(subset, batch_size=bs, shuffle=False)
    saliency = Saliency(lambda x: model(x))

    n_batches = 0
    n_examples = 0
    t0 = time.perf_counter()

    for images, labels in dl:
        if n_examples >= 40:
            break
        with torch.no_grad():
            logits = model(images)
            preds = logits.argmax(dim=1)

        # Saliency on batched input
        images_req = images.requires_grad_(True)
        # For batch saliency, target must be a list
        attrs = saliency.attribute(images_req, target=preds, abs=True)

        n_batches += 1
        n_examples += images.shape[0]

    elapsed = time.perf_counter() - t0
    tput = n_examples / elapsed
    throughputs.append(tput)
    print(f'  batch_size={bs:2d}: {tput:.1f} examples/sec')

# Plot throughput scaling
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(batch_sizes, throughputs, 'o-', color='#2196F3', linewidth=2, markersize=8)
ax.set_xlabel('Batch Size', fontsize=12)
ax.set_ylabel('Throughput (examples/sec)', fontsize=12)
ax.set_title('Saliency Attribution Throughput vs. Batch Size\n(ResNet-18, CPU)',
              fontweight='bold')
ax.grid(alpha=0.3)
for bs, tput in zip(batch_sizes, throughputs):
    ax.annotate(f'{tput:.0f}', (bs, tput), textcoords='offset points',
                  xytext=(0, 8), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('throughput_scaling.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved: throughput_scaling.png')

## Self-Check Questions

1. **Cache key collisions:** The cache uses `MD5(bytes + method + target)`. Is MD5 secure enough for this purpose? What does "secure" mean in a caching context vs. a cryptographic context?

2. **Completeness in reports:** The `attribution.completeness_satisfied` field checks `|delta| < 0.05`. Re-run `generate_attribution_report` with `n_steps=10` and `n_steps=200`. How does the delta change? What is the quality vs. latency trade-off?

3. **Throughput scaling:** Does throughput scale linearly with batch size? What limits scaling on CPU? What would change on GPU?

4. **Correct vs. incorrect attribution:** Look at the gallery from Step 6. Do incorrect predictions show different attribution patterns than correct ones? Do incorrect predictions concentrate attribution in the background more often?

5. **API integration:** The `generate_attribution_report` function returns a dict. Extend it to call the FastAPI service from Guide 02 (if running) instead of computing locally. What changes in the report structure?